# Main experiments notebook

**Research question 1:** Does an R-GCN model improve over non-graph baselines for predicting victim-
offender relationship categories? <br>

**Research question 2:** Which relation types contribute useful signal, and can removing relations improve
performance?

## Setup
As mentioned in the report, the experiments were conducted in the Jupyter server of the university. Please change the ```KG_PATH``` variable to the location of your ```kg_raw.ttl``` file.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import re
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

KG_PATH = Path('kg_raw.ttl')
OUT_DIR = Path('crime_kg_outputs')
OUT_DIR.mkdir(exist_ok=True)

print(KG_PATH.exists(), KG_PATH)

## Parsing the ontology

This parser used is a simple one that matches the generated ontology. There was no need to use RDFlib here.

In [3]:
URI_RE = re.compile(r'^<([^>]+)>\s+(?:<([^>]+)>|a)\s+(.+)\s+\.\s*$')
PREFIX_RE = re.compile(r'^:([A-Za-z_][A-Za-z0-9_]*)\s+(?:a|rdf:type)\s+owl:(Class|ObjectProperty|DatatypeProperty)')

def short(uri):
    if uri is None:
        return None
    return uri.rstrip('/#').split('/')[-1].split('#')[-1]

def parse_object(raw):
    raw = raw.strip()
    if raw.startswith('<'):
        return raw[1:raw.index('>')], 'uri'
    if raw.startswith('"'):
        end = raw.rfind('"')
        return raw[1:end], 'literal'
    return raw, 'other'

def parse_ttl(path):
    triples = []
    ontology_terms = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('@prefix') or line.startswith('#'):
                continue

            m_pref = PREFIX_RE.match(line)
            if m_pref:
                ontology_terms.append((m_pref.group(1), m_pref.group(2)))
                continue

            m = URI_RE.match(line)
            if not m:
                continue

            s = m.group(1)
            p = m.group(2) if m.group(2) is not None else 'rdf:type'
            o, o_kind = parse_object(m.group(3))
            triples.append({
                's': s,
                'p': p,
                'o': o,
                'o_kind': o_kind,
                's_short': short(s),
                'p_short': short(p),
                'o_short': short(o),
            })

    return pd.DataFrame(triples), pd.DataFrame(ontology_terms, columns=['term', 'kind'])

triples, ontology_terms = parse_ttl(KG_PATH)
print(triples.shape)
display(triples.head())
display(ontology_terms.head())

(38386, 7)


,s,p,o,o_kind,s_short,p_short,o_short
0,http://example.org/nibrs/incident/204216265,rdf:type,http://example.org/nibrs/ontology#Incident,uri,204216265,rdf:type,Incident
1,http://example.org/nibrs/incident/207870537,rdf:type,http://example.org/nibrs/ontology#Incident,uri,207870537,rdf:type,Incident
2,http://example.org/nibrs/incident/203959843,rdf:type,http://example.org/nibrs/ontology#Incident,uri,203959843,rdf:type,Incident
3,http://example.org/nibrs/incident/204243815,rdf:type,http://example.org/nibrs/ontology#Incident,uri,204243815,rdf:type,Incident
4,http://example.org/nibrs/incident/207861748,rdf:type,http://example.org/nibrs/ontology#Incident,uri,207861748,rdf:type,Incident


,term,kind
0,Incident,Class
1,Offense,Class
2,OffenseType,Class
3,Victim,Class
4,Offender,Class


## KG Diagnostics

This section checks whether the KG has enough relational structure for graph learning, since the ontology was mainly used for KG embeddings and logical rules, rather than GNNs in another project. The diagnostics are the relation distribution, node-type counts, and degree distribution. They are also used in the report as a finding - many entities are event-local, so the model can only benefit from message passing if the selected relations create useful shared context.


In [8]:
num_triples = len(triples)
num_object_triples = int((triples.o_kind == 'uri').sum())
num_literal_triples = int((triples.o_kind == 'literal').sum())

kg_size_summary = pd.DataFrame([
    {'statistic': 'triples', 'value': num_triples},
    {'statistic': 'object triples', 'value': num_object_triples},
    {'statistic': 'literal triples', 'value': num_literal_triples},
])
display(kg_size_summary)

relation_counts = (
    triples['p_short']
    .value_counts()
    .rename_axis('relation')
    .reset_index(name='count')
)
display(relation_counts)

type_triples = triples[(triples.p == 'rdf:type') & (triples.o_kind == 'uri')]
type_counts = (
    type_triples['o_short']
    .value_counts()
    .rename_axis('class')
    .reset_index(name='count')
)
display(type_counts)


,statistic,value
0,triples,38386
1,object triples,28091
2,literal triples,10295


,relation,count
0,rdf:type,8909
1,hasSex,3859
2,hasAge,3490
3,hasRelationshipType,2133
4,hasOffender,2133
5,hasVictim,2133
6,relatedToOffender,2131
7,hasVictimType,2068
8,victimInIncident,2068
9,involvedInOffense,1835


,class,count
0,VictimOffenderRelationship,2133
1,Victim,2068
2,Offender,1791
3,Offense,1445
4,Incident,1430
5,RelationshipType,30
6,VictimType,9
7,OffenseType,3


In [9]:
object_triples = triples[(triples.o_kind == 'uri') & (triples.p != 'rdf:type')].copy()
node_degree = Counter(object_triples['s']) + Counter(object_triples['o'])
degree_df = pd.DataFrame({
    'node': list(node_degree.keys()),
    'degree': list(node_degree.values()),
})
degree_df['prefix'] = degree_df['node'].map(lambda x: x.split('/')[-2] if '/' in x else 'unknown')

def get_type_map(triples):
    tm = {}
    for _, row in triples[(triples.p == 'rdf:type') & (triples.o_kind == 'uri')].iterrows():
        tm[row.s] = row.o_short
    return tm

type_map = get_type_map(triples)
degree_df['type'] = degree_df['node'].map(type_map).fillna(degree_df['prefix'])

degree_overall = degree_df['degree'].describe().to_frame().T
degree_overall.index = ['all entity nodes']
display(degree_overall)

degree_by_type = (
    degree_df
    .groupby('type')['degree']
    .describe()
    .sort_values('count', ascending=False)
)
display(degree_by_type)

degree_histogram = (
    degree_df['degree']
    .value_counts()
    .sort_index()
    .rename_axis('degree')
    .reset_index(name='num_nodes')
)
display(degree_histogram.head(20))

low_degree_share = (degree_df['degree'] <= 2).mean()
print(f'Share of entity nodes with degree <= 2: {low_degree_share:.3f}')


,count,mean,std,min,25%,50%,75%,max
all entity nodes,8902.0,4.309593,24.453089,1.0,3.0,3.0,4.0,1789.0


,count,mean,std,min,25%,50%,75%,max
type,,,,,,,,
VictimOffenderRelationship,2133.0,3.000000,0.000000,3.0,3.0,3.0,3.0,3.0
Victim,2068.0,4.949226,1.721498,2.0,4.0,5.0,5.0,21.0
Offender,1791.0,3.380793,1.792333,1.0,3.0,3.0,3.0,35.0
Offense,1445.0,3.269896,0.772063,3.0,3.0,3.0,3.0,19.0
Incident,1430.0,3.709091,1.243312,3.0,3.0,3.0,4.0,19.0
RelationshipType,25.0,85.320000,171.355799,1.0,3.0,29.0,59.0,779.0
VictimType,7.0,295.428571,660.625173,2.0,3.5,55.0,107.5,1789.0
OffenseType,3.0,481.666667,517.263311,17.0,203.0,389.0,714.0,1039.0


,degree,num_nodes
0,1,208
1,2,99
2,3,5742
3,4,634
4,5,1610
5,6,98
6,7,319
7,8,29
8,9,76
9,10,4


Share of entity nodes with degree <= 2: 0.034


## Predicting relationship type classification

The target edge: `(VictimOffenderRelationship, hasRelationshipType, RelationshipType)`.

To avoid leakage we remove all ```hasRelationshipType``` edges from the GNN message graph before training the node classifier. The model can use the relationship record's victim/offender links and the surrounding incident/offense context, but not the answer edge.

In [ ]:
def relation_map(df, pred_short):
    sub = df[(df.p_short == pred_short) & (df.o_kind == 'uri')]
    return dict(zip(sub.s, sub.o))

rel_has_victim = relation_map(triples, 'hasVictim')
rel_has_offender = relation_map(triples, 'hasOffender')
rel_has_type = relation_map(triples, 'hasRelationshipType')

records = sorted(set(rel_has_victim) & set(rel_has_offender) & set(rel_has_type))
task_df = pd.DataFrame({
    'record': records,
    'victim': [rel_has_victim[r] for r in records],
    'offender': [rel_has_offender[r] for r in records],
    'relationship_type': [rel_has_type[r] for r in records],
})
task_df['relationship_id'] = task_df['relationship_type'].map(short).astype(str)

rel_lookup_path = Path('WA-2024/NIBRS_RELATIONSHIP.csv')
if rel_lookup_path.exists():
    rel_lookup = pd.read_csv(rel_lookup_path)
    rel_lookup['relationship_id'] = rel_lookup['relationship_id'].astype(str)
    id_to_name = dict(zip(rel_lookup.relationship_id, rel_lookup.relationship_name))
    id_to_code = dict(zip(rel_lookup.relationship_id, rel_lookup.relationship_code))
else:
    id_to_name = {}
    id_to_code = {}

task_df['relationship_name'] = task_df['relationship_id'].map(id_to_name).fillna(task_df['relationship_id'])
task_df['relationship_code'] = task_df['relationship_id'].map(id_to_code).fillna(task_df['relationship_id'])

# We group the large number of relationship labels into a smaller number of more general categories for classification
INTIMATE = {'3', '6', '12', '21', '26', '27', '28'}
FAMILY = {'2', '4', '5', '10', '11', '13', '15', '17', '19', '20', '22', '23', '29', '30'}
ACQUAINTANCE = {'1', '7', '8', '9', '16'}
STRANGER = {'24'}
UNKNOWN = {'18'}
OTHER = {'25'}

def coarse_relationship(label):
    label = str(label)
    if label in UNKNOWN:
        return 'unknown'
    if label in STRANGER:
        return 'stranger'
    if label in INTIMATE:
        return 'intimate_partner_or_spouse'
    if label in FAMILY:
        return 'family'
    if label in ACQUAINTANCE:
        return 'acquaintance_or_known'
    if label in OTHER:
        return 'victim_was_offender'
    return 'other'

task_df['relationship_fine'] = task_df['relationship_id']
task_df['relationship_coarse'] = task_df['relationship_id'].map(coarse_relationship)

print('Relationship records:', len(task_df))
display(task_df.head())
print('Fine label distribution:')
display(task_df[['relationship_id', 'relationship_code', 'relationship_name']].value_counts().rename('count').reset_index())
print('Coarse label distribution:')
display(task_df['relationship_coarse'].value_counts().rename_axis('label').reset_index(name='count'))


Relationship records: 2133


,record,victim,offender,relationship_type,relationship_id,relationship_name,relationship_code,relationship_fine,relationship_coarse
0,http://example.org/nibrs/victim-offender-relat...,http://example.org/nibrs/victim/225644744,http://example.org/nibrs/offender/230489002,http://example.org/nibrs/relationship/18,18,18,18,18,unknown
1,http://example.org/nibrs/victim-offender-relat...,http://example.org/nibrs/victim/225645368,http://example.org/nibrs/offender/230489630,http://example.org/nibrs/relationship/3,3,3,3,3,intimate_partner_or_spouse
2,http://example.org/nibrs/victim-offender-relat...,http://example.org/nibrs/victim/226192937,http://example.org/nibrs/offender/231032169,http://example.org/nibrs/relationship/19,19,19,19,19,family
3,http://example.org/nibrs/victim-offender-relat...,http://example.org/nibrs/victim/227349465,http://example.org/nibrs/offender/232184838,http://example.org/nibrs/relationship/7,7,7,7,7,acquaintance_or_known
4,http://example.org/nibrs/victim-offender-relat...,http://example.org/nibrs/victim/227636308,http://example.org/nibrs/offender/232472368,http://example.org/nibrs/relationship/14,14,14,14,14,other


Fine label distribution:


,relationship_id,relationship_code,relationship_name,count
0,18,18,18,779
1,24,24,24,422
2,1,1,1,171
3,16,16,16,168
4,3,3,3,158
5,27,27,27,61
6,21,21,21,59
7,5,5,5,56
8,17,17,17,55
9,9,9,9,49


Coarse label distribution:


,label,count
0,unknown,779
1,stranger,422
2,acquaintance_or_known,394
3,intimate_partner_or_spouse,290
4,family,191
5,other,32
6,victim_was_offender,25


In [ ]:
# We remove Relationship Unknown and predict the defined relationship groups
task_df = task_df[task_df['relationship_coarse'] != 'unknown'].reset_index(drop=True)
task_df['relationship_label'] = task_df['relationship_coarse']

# Stratified split for imbalanced labels
min_per_class = 5
label_counts = task_df['relationship_label'].value_counts()
kept_labels = label_counts[label_counts >= min_per_class].index
task_df = task_df[task_df['relationship_label'].isin(kept_labels)].reset_index(drop=True)

rng = np.random.default_rng(SEED)
train_parts, val_parts, test_parts = [], [], []

for label, group in task_df.groupby('relationship_label').groups.items():
    idx = np.array(list(group))
    rng.shuffle(idx)
    n = len(idx)

    n_test = max(1, int(round(0.15 * n)))
    n_val = max(1, int(round(0.15 * n)))

    if n_test + n_val > n - 1:
        n_test = 1
        n_val = 1

    test_parts.append(idx[:n_test])
    val_parts.append(idx[n_test:n_test + n_val])
    train_parts.append(idx[n_test + n_val:])

train_idx = np.concatenate(train_parts)
val_idx = np.concatenate(val_parts)
test_idx = np.concatenate(test_parts)

rng.shuffle(train_idx)
rng.shuffle(val_idx)
rng.shuffle(test_idx)

print('Task variant:', 'coarse_known')
print('Split sizes:', len(train_idx), len(val_idx), len(test_idx))
print('Classes:', task_df.relationship_label.nunique())
print('Label distribution after filtering:')
display(task_df['relationship_label'].value_counts().rename_axis('label').reset_index(name='count'))
print('Minimum split counts:')
print(pd.DataFrame({
    'train': task_df.iloc[train_idx].relationship_label.value_counts(),
    'val': task_df.iloc[val_idx].relationship_label.value_counts(),
    'test': task_df.iloc[test_idx].relationship_label.value_counts(),
}).fillna(0).astype(int).min())


Task variant: coarse_known
Split sizes: 946 204 204
Classes: 6
Label distribution after filtering:


,label,count
0,stranger,422
1,acquaintance_or_known,394
2,intimate_partner_or_spouse,290
3,family,191
4,other,32
5,victim_was_offender,25


Minimum split counts:
train    17
val       4
test      4
dtype: int64


## Baselines
 
 In this section we define the majority and the tabular logistic regression baselines that will be used to compare against the R-GCN model.

In [7]:
y = task_df['relationship_label'].to_numpy()
majority_label = pd.Series(y[train_idx]).mode().iloc[0]
majority_pred = np.array([majority_label] * len(test_idx))

print('Majority label:', majority_label)
print('Majority accuracy:', accuracy_score(y[test_idx], majority_pred))
print('Majority macro-F1:', f1_score(y[test_idx], majority_pred, average='macro'))

Majority label: stranger
Majority accuracy: 0.3088235294117647
Majority macro-F1: 0.07865168539325842


In [8]:
literal_df = triples[triples.o_kind == 'literal'].copy()
literal_lookup = {(r.s, r.p_short): r.o for r in literal_df.itertuples()}

victim_incident = relation_map(triples, 'victimInIncident')
offender_incident = relation_map(triples, 'offenderInIncident')
victim_type = relation_map(triples, 'hasVictimType')

victim_offenses = defaultdict(list)
for r in triples[(triples.p_short == 'involvedInOffense') & (triples.o_kind == 'uri')].itertuples():
    victim_offenses[r.s].append(r.o)

offense_type = relation_map(triples, 'hasOffenseType')
incident_hour = {s: o for (s, p), o in literal_lookup.items() if p == 'hasHour'}

def make_tabular_features(task_df):
    rows = []
    for row in task_df.itertuples():
        v, o = row.victim, row.offender
        inc = victim_incident.get(v) or offender_incident.get(o)
        offenses = victim_offenses.get(v, [])
        offense_types = sorted(short(offense_type[x]) for x in offenses if x in offense_type)
        rows.append({
            'victim_type': short(victim_type.get(v)),
            'victim_sex': literal_lookup.get((v, 'hasSex')),
            'offender_sex': literal_lookup.get((o, 'hasSex')),
            'victim_age': pd.to_numeric(literal_lookup.get((v, 'hasAge')), errors='coerce'),
            'offender_age': pd.to_numeric(literal_lookup.get((o, 'hasAge')), errors='coerce'),
            'incident_hour': pd.to_numeric(incident_hour.get(inc), errors='coerce'),
            'num_victim_offenses': len(offenses),
            'first_offense_type': offense_types[0] if offense_types else None,
        })
    return pd.DataFrame(rows)

X_tab = make_tabular_features(task_df)
display(X_tab.head())

,victim_type,victim_sex,offender_sex,victim_age,offender_age,incident_hour,num_victim_offenses,first_offense_type
0,4,M,F,27.0,27,1.0,1,13A
1,4,M,M,28.0,28,NaN,1,13A
2,5,M,M,0.0,23,12.0,1,13A
3,4,M,M,46.0,40,23.0,1,13A
4,4,M,M,12.0,16,20.0,1,13A


In [9]:
cat_cols = ['victim_type', 'victim_sex', 'offender_sex', 'first_offense_type']
num_cols = ['victim_age', 'offender_age', 'incident_hour', 'num_victim_offenses']

preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ('num', SimpleImputer(strategy='median'), num_cols),
])

tabular_clf = make_pipeline(
    preprocess,
    LogisticRegression(max_iter=2000, class_weight='balanced', n_jobs=None),
)
tabular_clf.fit(X_tab.iloc[train_idx], y[train_idx])
tab_pred = tabular_clf.predict(X_tab.iloc[test_idx])

print('Tabular accuracy:', accuracy_score(y[test_idx], tab_pred))
print('Tabular macro-F1:', f1_score(y[test_idx], tab_pred, average='macro'))
print(classification_report(y[test_idx], tab_pred, zero_division=0))

Tabular accuracy: 0.37254901960784315
Tabular macro-F1: 0.3125527842022688
                            precision    recall  f1-score   support

     acquaintance_or_known       0.53      0.17      0.26        59
                    family       0.32      0.21      0.25        29
intimate_partner_or_spouse       0.52      0.70      0.60        44
                     other       0.09      0.60      0.16         5
                  stranger       0.68      0.37      0.47        63
       victim_was_offender       0.07      0.75      0.14         4

                  accuracy                           0.37       204
                 macro avg       0.37      0.47      0.31       204
              weighted avg       0.52      0.37      0.39       204



/opt/micromamba/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## R-GCN model

In [10]:
message_triples = triples[
    (triples.o_kind == 'uri')
    & (triples.p != 'rdf:type')
    & (triples.p_short != 'hasRelationshipType')
].copy()

nodes = sorted(set(message_triples.s) | set(message_triples.o) | set(task_df.record))
node_to_id = {n: i for i, n in enumerate(nodes)}

base_relations = sorted(message_triples.p_short.unique())
rel_to_id = {r: i for i, r in enumerate(base_relations)}

src, dst, rel = [], [], []
for row in message_triples.itertuples():
    if row.s not in node_to_id or row.o not in node_to_id:
        continue
    r = rel_to_id[row.p_short]
    s = node_to_id[row.s]
    o = node_to_id[row.o]
    src.append(s); dst.append(o); rel.append(r)
    src.append(o); dst.append(s); rel.append(r + len(base_relations))

edge_src = torch.tensor(src, dtype=torch.long)
edge_dst = torch.tensor(dst, dtype=torch.long)
edge_rel = torch.tensor(rel, dtype=torch.long)
num_relations = 2 * len(base_relations)

label_nodes = torch.tensor([node_to_id[r] for r in task_df.record], dtype=torch.long)
label_to_id = {lab: i for i, lab in enumerate(sorted(task_df.relationship_label.unique()))}
id_to_label = {i: lab for lab, i in label_to_id.items()}
labels = torch.tensor([label_to_id[x] for x in task_df.relationship_label], dtype=torch.long)

train_label_counts = np.bincount(labels[train_idx].numpy(), minlength=len(label_to_id))
class_weights = train_label_counts.sum() / np.maximum(train_label_counts, 1)
class_weights = class_weights / class_weights.mean()
class_weights = torch.tensor(class_weights, dtype=torch.float)
print('Class weights:', {id_to_label[i]: round(float(w), 3) for i, w in enumerate(class_weights)})

train_idx_t = torch.tensor(train_idx, dtype=torch.long)
val_idx_t = torch.tensor(val_idx, dtype=torch.long)
test_idx_t = torch.tensor(test_idx, dtype=torch.long)

print('Nodes:', len(nodes))
print('Edges:', edge_src.numel())
print('Relations incl. inverse:', num_relations)
print('Classes:', len(label_to_id))


Class weights: {'acquaintance_or_known': 0.176, 'family': 0.365, 'intimate_partner_or_spouse': 0.24, 'other': 2.204, 'stranger': 0.164, 'victim_was_offender': 2.852}
Nodes: 8877
Edges: 34098
Relations incl. inverse: 18
Classes: 6


In [11]:
class RGCNLayer(nn.Module):
    def __init__(self, hidden_dim, num_relations, dropout=0.1):
        super().__init__()
        self.rel_weight = nn.Parameter(torch.empty(num_relations, hidden_dim, hidden_dim))
        self.self_loop = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.rel_weight)

    def forward(self, x, edge_src, edge_dst, edge_rel, num_nodes):
        out = torch.zeros_like(x)
        msg = torch.bmm(x[edge_src].unsqueeze(1), self.rel_weight[edge_rel]).squeeze(1)
        out.index_add_(0, edge_dst, msg)

        deg = torch.zeros(num_nodes, device=x.device)
        deg.index_add_(0, edge_dst, torch.ones_like(edge_dst, dtype=torch.float, device=x.device))
        out = out / deg.clamp(min=1).unsqueeze(-1)
        out = out + self.self_loop(x)
        out = F.relu(out)
        return self.dropout(out)

class RGCNNodeClassifier(nn.Module):
    def __init__(self, num_nodes, num_relations, hidden_dim, num_classes, num_layers=2, dropout=0.1):
        super().__init__()
        self.node_emb = nn.Embedding(num_nodes, hidden_dim)
        self.layers = nn.ModuleList([
            RGCNLayer(hidden_dim, num_relations, dropout=dropout)
            for _ in range(num_layers)
        ])
        self.classifier = nn.Linear(hidden_dim, num_classes)
        nn.init.xavier_uniform_(self.node_emb.weight)

    def forward(self, edge_src, edge_dst, edge_rel):
        x = self.node_emb.weight
        num_nodes = x.size(0)
        for layer in self.layers:
            x = layer(x, edge_src, edge_dst, edge_rel, num_nodes)
        return self.classifier(x), x

def evaluate_logits(logits, split_idx):
    pred = logits[label_nodes[split_idx]].argmax(dim=-1).detach().cpu().numpy()
    true = labels[split_idx].detach().cpu().numpy()
    return {
        'accuracy': accuracy_score(true, pred),
        'macro_f1': f1_score(true, pred, average='macro'),
    }

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
edge_src_d = edge_src.to(device)
edge_dst_d = edge_dst.to(device)
edge_rel_d = edge_rel.to(device)
label_nodes_d = label_nodes.to(device)
labels_d = labels.to(device)
class_weights_d = class_weights.to(device)
train_idx_d = train_idx_t.to(device)
val_idx_d = val_idx_t.to(device)
test_idx_d = test_idx_t.to(device)

label_nodes = label_nodes_d
labels = labels_d

model = RGCNNodeClassifier(
    num_nodes=len(nodes),
    num_relations=num_relations,
    hidden_dim=64,
    num_classes=len(label_to_id),
    num_layers=2,
    dropout=0.2,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)

best_val = -1
best_state = None
history = []

for epoch in range(1, 301):
    model.train()
    optimizer.zero_grad()
    logits, emb = model(edge_src_d, edge_dst_d, edge_rel_d)
    loss = F.cross_entropy(
        logits[label_nodes[train_idx_d]],
        labels[train_idx_d],
        weight=class_weights_d,
    )
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0 or epoch == 1:
        model.eval()
        with torch.no_grad():
            logits, emb = model(edge_src_d, edge_dst_d, edge_rel_d)
            train_metrics = evaluate_logits(logits, train_idx_d)
            val_metrics = evaluate_logits(logits, val_idx_d)
        history.append({'epoch': epoch, 'loss': float(loss.item()), **{f'train_{k}': v for k, v in train_metrics.items()}, **{f'val_{k}': v for k, v in val_metrics.items()}})
        if val_metrics['macro_f1'] > best_val:
            best_val = val_metrics['macro_f1']
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if epoch % 50 == 0 or epoch == 1:
            print(epoch, 'loss', round(loss.item(), 4), 'val', val_metrics)

history_df = pd.DataFrame(history)
display(history_df.tail())


1 loss 1.7976 val {'accuracy': 0.28921568627450983, 'macro_f1': 0.07477820025348543}
50 loss 0.0053 val {'accuracy': 0.5049019607843137, 'macro_f1': 0.322983443061281}
100 loss 0.0017 val {'accuracy': 0.43137254901960786, 'macro_f1': 0.19907184378713427}
150 loss 0.0012 val {'accuracy': 0.36764705882352944, 'macro_f1': 0.1574646411733892}
200 loss 0.0009 val {'accuracy': 0.4068627450980392, 'macro_f1': 0.1782075275525057}
250 loss 0.0012 val {'accuracy': 0.30392156862745096, 'macro_f1': 0.09571749097649009}
300 loss 0.0008 val {'accuracy': 0.3137254901960784, 'macro_f1': 0.10468691413573304}


,epoch,loss,train_accuracy,train_macro_f1,val_accuracy,val_macro_f1
26,260,0.001668,1.0,1.0,0.299020,0.090378
27,270,0.001252,1.0,1.0,0.441176,0.192131
28,280,0.001841,1.0,1.0,0.313725,0.105291
29,290,0.000943,1.0,1.0,0.308824,0.100874
30,300,0.000816,1.0,1.0,0.313725,0.104687


In [13]:
model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
model.eval()
with torch.no_grad():
    logits, emb = model(edge_src_d, edge_dst_d, edge_rel_d)
    test_metrics = evaluate_logits(logits, test_idx_d)

pred = logits[label_nodes[test_idx_d]].argmax(dim=-1).detach().cpu().numpy()
true = labels[test_idx_d].detach().cpu().numpy()
pred_labels = [id_to_label[i] for i in pred]
true_labels = [id_to_label[i] for i in true]

print('R-GCN test:', test_metrics)
print(classification_report(true_labels, pred_labels, zero_division=0))

R-GCN test: {'accuracy': 0.5147058823529411, 'macro_f1': 0.35044571130370933}
                            precision    recall  f1-score   support

     acquaintance_or_known       0.62      0.54      0.58        59
                    family       0.25      0.03      0.06        29
intimate_partner_or_spouse       0.39      0.98      0.55        44
                     other       1.00      0.20      0.33         5
                  stranger       0.82      0.44      0.58        63
       victim_was_offender       0.00      0.00      0.00         4

                  accuracy                           0.51       204
                 macro avg       0.51      0.37      0.35       204
              weighted avg       0.58      0.51      0.48       204



In [ ]:
def build_rgcn_graph(remove_relations=()):
    remove_relations = set(remove_relations)
    df = triples[
        (triples.o_kind == 'uri')
        & (triples.p != 'rdf:type')
        & (triples.p_short != 'hasRelationshipType')
        & (~triples.p_short.isin(remove_relations))
    ].copy()

    nodes_local = sorted(set(df.s) | set(df.o) | set(task_df.record))
    node_to_id_local = {n: i for i, n in enumerate(nodes_local)}

    base_relations_local = sorted(df.p_short.unique())
    rel_to_id_local = {r: i for i, r in enumerate(base_relations_local)}

    src, dst, rel = [], [], []
    for row in df.itertuples():
        if row.s not in node_to_id_local or row.o not in node_to_id_local:
            continue
        r = rel_to_id_local[row.p_short]
        s = node_to_id_local[row.s]
        o = node_to_id_local[row.o]
        src.append(s); dst.append(o); rel.append(r)
        src.append(o); dst.append(s); rel.append(r + len(base_relations_local))

    return {
        'name': ','.join(sorted(remove_relations)) if remove_relations else 'full',
        'message_triples': df,
        'nodes': nodes_local,
        'node_to_id': node_to_id_local,
        'edge_src': torch.tensor(src, dtype=torch.long),
        'edge_dst': torch.tensor(dst, dtype=torch.long),
        'edge_rel': torch.tensor(rel, dtype=torch.long),
        'num_relations': 2 * len(base_relations_local),
        'base_relations': base_relations_local,
    }


def train_one_rgcn_variant(
    variant_name,
    remove_relations=(),
    seeds=(1, 2, 3),
    hidden_dim=64,
    num_layers=2,
    epochs=200,
    lr=0.01,
    weight_decay=1e-4,
    dropout=0.2,
):
    graph = build_rgcn_graph(remove_relations)
    local_label_to_id = {lab: i for i, lab in enumerate(sorted(task_df.relationship_label.unique()))}
    local_id_to_label = {i: lab for lab, i in local_label_to_id.items()}

    label_nodes_cpu = torch.tensor([graph['node_to_id'][r] for r in task_df.record], dtype=torch.long)
    labels_cpu = torch.tensor([local_label_to_id[x] for x in task_df.relationship_label], dtype=torch.long)

    rows = []
    for seed in seeds:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        edge_src_d = graph['edge_src'].to(device)
        edge_dst_d = graph['edge_dst'].to(device)
        edge_rel_d = graph['edge_rel'].to(device)
        label_nodes_d = label_nodes_cpu.to(device)
        labels_d = labels_cpu.to(device)
        train_idx_d = torch.tensor(train_idx, dtype=torch.long, device=device)
        val_idx_d = torch.tensor(val_idx, dtype=torch.long, device=device)
        test_idx_d = torch.tensor(test_idx, dtype=torch.long, device=device)

        train_label_counts = np.bincount(labels_cpu[train_idx].numpy(), minlength=len(local_label_to_id))
        weights = train_label_counts.sum() / np.maximum(train_label_counts, 1)
        weights = weights / weights.mean()
        weights_d = torch.tensor(weights, dtype=torch.float, device=device)

        model = RGCNNodeClassifier(
            num_nodes=len(graph['nodes']),
            num_relations=graph['num_relations'],
            hidden_dim=hidden_dim,
            num_classes=len(local_label_to_id),
            num_layers=num_layers,
            dropout=dropout,
        ).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

        best_val = -1.0
        best_state = None
        for epoch in range(1, epochs + 1):
            model.train()
            optimizer.zero_grad()
            logits, _ = model(edge_src_d, edge_dst_d, edge_rel_d)
            loss = F.cross_entropy(
                logits[label_nodes_d[train_idx_d]],
                labels_d[train_idx_d],
                weight=weights_d,
            )
            loss.backward()
            optimizer.step()

            if epoch % 10 == 0 or epoch == epochs:
                model.eval()
                with torch.no_grad():
                    logits, _ = model(edge_src_d, edge_dst_d, edge_rel_d)
                    val_pred = logits[label_nodes_d[val_idx_d]].argmax(dim=-1).detach().cpu().numpy()
                    val_true = labels_d[val_idx_d].detach().cpu().numpy()
                    val_macro_f1 = f1_score(val_true, val_pred, average='macro')
                if val_macro_f1 > best_val:
                    best_val = val_macro_f1
                    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
        model.eval()
        with torch.no_grad():
            logits, _ = model(edge_src_d, edge_dst_d, edge_rel_d)
            test_pred = logits[label_nodes_d[test_idx_d]].argmax(dim=-1).detach().cpu().numpy()
            test_true = labels_d[test_idx_d].detach().cpu().numpy()

        rows.append({
            'variant': variant_name,
            'removed_relations': ', '.join(remove_relations) if remove_relations else 'none',
            'seed': seed,
            'num_nodes': len(graph['nodes']),
            'num_edges': int(graph['edge_src'].numel()),
            'num_base_relations': len(graph['base_relations']),
            'accuracy': accuracy_score(test_true, test_pred),
            'macro_f1': f1_score(test_true, test_pred, average='macro'),
            'weighted_f1': f1_score(test_true, test_pred, average='weighted'),
            'best_val_macro_f1': best_val,
        })

    return pd.DataFrame(rows)


ABLATION_SEEDS = [1, 2, 3] 
ABLATION_EPOCHS = 200

ablation_specs = [
    ('R-GCN full graph', []),
    ('R-GCN no relatedToOffender', ['relatedToOffender']),
    ('R-GCN no involvedInOffense', ['involvedInOffense']),
    ('R-GCN no hasOffenseType', ['hasOffenseType']),
    ('R-GCN no offense context', ['involvedInOffense', 'hasOffenseType']),
    ('R-GCN demographics/core only', ['relatedToOffender', 'involvedInOffense', 'hasOffenseType']),
    ('R-GCN no victimInIncident', ['victimInIncident']),
    ('R-GCN no offenderInIncident', ['offenderInIncident']),
    ('R-GCN no hasVictimType', ['hasVictimType'])
]

ablation_results = []
for name, removed in ablation_specs:
    print(f'Running {name} | removed={removed}')
    result = train_one_rgcn_variant(
        name,
        remove_relations=removed,
        seeds=ABLATION_SEEDS,
        epochs=ABLATION_EPOCHS,
    )
    ablation_results.append(result)

ablation_results_df = pd.concat(ablation_results, ignore_index=True)
ablation_results_df.to_csv(OUT_DIR / 'rgcn_ablation_results.csv', index=False)
display(ablation_results_df)

ablation_summary = (
    ablation_results_df
    .groupby(['variant', 'removed_relations'], as_index=False)
    .agg(
        accuracy_mean=('accuracy', 'mean'),
        accuracy_std=('accuracy', 'std'),
        macro_f1_mean=('macro_f1', 'mean'),
        macro_f1_std=('macro_f1', 'std'),
        weighted_f1_mean=('weighted_f1', 'mean'),
        weighted_f1_std=('weighted_f1', 'std'),
        num_edges=('num_edges', 'first'),
        num_base_relations=('num_base_relations', 'first'),
    )
    .sort_values('macro_f1_mean', ascending=False)
)

display(ablation_summary)



Running R-GCN full graph | removed=[]
Running R-GCN no relatedToOffender | removed=['relatedToOffender']
Running R-GCN no involvedInOffense | removed=['involvedInOffense']
Running R-GCN no hasOffenseType | removed=['hasOffenseType']
Running R-GCN no offense context | removed=['involvedInOffense', 'hasOffenseType']
Running R-GCN demographics/core only | removed=['relatedToOffender', 'involvedInOffense', 'hasOffenseType']
Running R-GCN no victimInIncident | removed=['victimInIncident']
Running R-GCN no offenderInIncident | removed=['offenderInIncident']
Running R-GCN no hasVictimType | removed=['hasVictimType']


,variant,removed_relations,seed,num_nodes,num_edges,num_base_relations,accuracy,macro_f1,weighted_f1,best_val_macro_f1
0,R-GCN full graph,none,1,8877,34098,9,0.544118,0.405353,0.542294,0.380670
1,R-GCN full graph,none,2,8877,34098,9,0.544118,0.394223,0.537576,0.348935
2,R-GCN full graph,none,3,8877,34098,9,0.485294,0.314828,0.414928,0.273587
3,R-GCN no relatedToOffender,relatedToOffender,1,8877,29836,8,0.563725,0.423005,0.566476,0.389578
4,R-GCN no relatedToOffender,relatedToOffender,2,8877,29836,8,0.539216,0.369576,0.519154,0.338203
5,R-GCN no relatedToOffender,relatedToOffender,3,8877,29836,8,0.480392,0.312415,0.410742,0.272645
6,R-GCN no involvedInOffense,involvedInOffense,1,8877,30428,8,0.558824,0.419482,0.560233,0.390262
7,R-GCN no involvedInOffense,involvedInOffense,2,8877,30428,8,0.504902,0.340015,0.477725,0.333416
8,R-GCN no involvedInOffense,involvedInOffense,3,8877,30428,8,0.460784,0.336679,0.428220,0.307916
9,R-GCN no hasOffenseType,hasOffenseType,1,8874,31208,8,0.509804,0.366619,0.499538,0.365824


,variant,removed_relations,accuracy_mean,accuracy_std,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std,num_edges,num_base_relations
0,R-GCN demographics/core only,"relatedToOffender, involvedInOffense, hasOffen...",0.544118,0.008490,0.399947,0.003987,0.543596,0.007873,23276,6
5,R-GCN no offenderInIncident,offenderInIncident,0.531046,0.045547,0.378723,0.056424,0.508189,0.079430,30516,8
1,R-GCN full graph,none,0.524510,0.033962,0.371468,0.049366,0.498266,0.072212,34098,9
7,R-GCN no relatedToOffender,relatedToOffender,0.527778,0.042828,0.368332,0.055305,0.498790,0.079839,29836,8
6,R-GCN no offense context,"involvedInOffense, hasOffenseType",0.517974,0.042828,0.366988,0.047306,0.500675,0.072694,27538,7
8,R-GCN no victimInIncident,victimInIncident,0.519608,0.055676,0.365810,0.072828,0.493679,0.090783,29962,8
4,R-GCN no involvedInOffense,involvedInOffense,0.508170,0.049101,0.365392,0.046873,0.488726,0.066690,30428,8
2,R-GCN no hasOffenseType,hasOffenseType,0.493464,0.060502,0.363595,0.043574,0.491738,0.056997,31208,8
3,R-GCN no hasVictimType,hasVictimType,0.498366,0.027874,0.347099,0.035210,0.467304,0.056048,29962,8


In [18]:
for rel in [
    "relatedToOffender",
    "involvedInOffense",
    "hasOffenseType",
    "victimInIncident",
    "offenderInIncident",
    "hasVictimType",
]:
    print(rel, (message_triples["p_short"] == rel).sum())

relatedToOffender 2131
involvedInOffense 1835
hasOffenseType 1445
victimInIncident 2068
offenderInIncident 1791
hasVictimType 2068


## Additional ablations

In [14]:
def summarize_edge_set(df, name):
    rel_counts = df.p_short.value_counts()
    print(name, 'triples:', len(df), 'relations:', rel_counts.shape[0])
    display(rel_counts.rename_axis('relation').reset_index(name='count'))

summarize_edge_set(message_triples, 'Message graph')

for remove in [['relatedToOffender'], ['structurallySimilarTo'], ['relatedToOffender', 'structurallySimilarTo']]:
    ablated = message_triples[~message_triples.p_short.isin(remove)]
    summarize_edge_set(ablated, f'Ablated {remove}')

Message graph triples: 17049 relations: 9


,relation,count
0,hasVictim,2133
1,hasOffender,2133
2,relatedToOffender,2131
3,victimInIncident,2068
4,hasVictimType,2068
5,involvedInOffense,1835
6,offenderInIncident,1791
7,hasOffenseType,1445
8,hasIncident,1445


Ablated ['relatedToOffender'] triples: 14918 relations: 8


,relation,count
0,hasVictim,2133
1,hasOffender,2133
2,victimInIncident,2068
3,hasVictimType,2068
4,involvedInOffense,1835
5,offenderInIncident,1791
6,hasOffenseType,1445
7,hasIncident,1445


Ablated ['structurallySimilarTo'] triples: 17049 relations: 9


,relation,count
0,hasVictim,2133
1,hasOffender,2133
2,relatedToOffender,2131
3,victimInIncident,2068
4,hasVictimType,2068
5,involvedInOffense,1835
6,offenderInIncident,1791
7,hasOffenseType,1445
8,hasIncident,1445


Ablated ['relatedToOffender', 'structurallySimilarTo'] triples: 14918 relations: 8


,relation,count
0,hasVictim,2133
1,hasOffender,2133
2,victimInIncident,2068
3,hasVictimType,2068
4,involvedInOffense,1835
5,offenderInIncident,1791
6,hasOffenseType,1445
7,hasIncident,1445


## Export Splits for the ULTRA prediction

This section creates standard KG completion files. The target query is:

`(VictimOffenderRelationship, hasRelationshipType, ?)`

We use these TSV files in ```crime_kg_ultra.ipynb```. 

In [ ]:
kg_object = triples[(triples.o_kind == 'uri') & (triples.p != 'rdf:type')].copy()

target_edges = kg_object[kg_object.p_short == 'hasRelationshipType'].copy()
target_edges = target_edges[target_edges.s.isin(set(task_df.record))]

train_records = set(task_df.iloc[train_idx].record)
val_records = set(task_df.iloc[val_idx].record)
test_records = set(task_df.iloc[test_idx].record)

train_target = target_edges[target_edges.s.isin(train_records)]
val_target = target_edges[target_edges.s.isin(val_records)]
test_target = target_edges[target_edges.s.isin(test_records)]

non_target = kg_object[kg_object.p_short != 'hasRelationshipType']
ultra_train = pd.concat([non_target, train_target], ignore_index=True)

def write_kg_tsv(df, path):
    out = df[['s', 'p_short', 'o']].copy()
    out.to_csv(path, sep='\t', header=False, index=False)
    print(path, len(out))

write_kg_tsv(ultra_train, OUT_DIR / 'ultra_train.tsv')
write_kg_tsv(val_target, OUT_DIR / 'ultra_valid.tsv')
write_kg_tsv(test_target, OUT_DIR / 'ultra_test.tsv')
write_kg_tsv(non_target, OUT_DIR / 'strict_message_graph.tsv')

crime_kg_outputs/ultra_train.tsv 17995
crime_kg_outputs/ultra_valid.tsv 204
crime_kg_outputs/ultra_test.tsv 204
crime_kg_outputs/strict_message_graph.tsv 17049
